# 02 — RetNet Architecture Lab

Prototype and explore the **RetNet** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.retnet.config import RetNetConfig
from src.models.retnet.model import RetNetSLM
from src.core.generation import generate
from src.utils.training import count_params

In [2]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = RetNetConfig(
    vocab_size=50257, block_size=32,
    n_layer=2, n_head=2, n_embd=64,
    intermediate_size=128, dropout=0.0,
)
model = RetNetSLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

Parameters: 3,298,944


In [3]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

logits: torch.Size([2, 32, 50257])  |  loss: 10.8227


In [4]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

Input tokens:  [46772, 49979, 35110, 38905]
Output tokens: [46772, 49979, 35110, 38905, 20013, 5089, 335, 6802, 7220, 34122, 29627, 34444, 9131, 18211]


In [5]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'retnet_tiny':   RetNetConfig(vocab_size=50257, block_size=16,  n_layer=2,  n_head=2,  n_embd=64,  intermediate_size=128),
    'retnet_small':  RetNetConfig(vocab_size=50257, block_size=128, n_layer=6,  n_head=6,  n_embd=384, intermediate_size=1024),
    'retnet_medium': RetNetConfig(vocab_size=50257, block_size=256, n_layer=12, n_head=12, n_embd=768, intermediate_size=2048),
}
for name, cfg in configs.items():
    n = count_params(RetNetSLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')

retnet_tiny   :    3,298,944 params  (3.3M)
retnet_small  :   29,925,120 params  (29.9M)
retnet_medium :  123,569,664 params  (123.6M)
